# 1. Classical: find the exact answers

Run this notebook, then [quantum](quantum_capacity_constrained_clustering.ipynb),
then [results](results.ipynb), using the `.venv` kernel.
We run **all supplied demand vectors** from the selected S or M input file.
S has (N,K,Q)=(4,2,7); M has (10,3,20). The fixed `s-0` example below
illustrates S, while the executable tables use the selected tier.
Core algorithms stay inline; shared helpers handle experiment bookkeeping,
inputs, files and QUBO checks. There is no command-line runner.

## The clustering problem (A1)

Group nearby customers into vehicles. Every customer is served once;
each vehicle's total demand is at most $Q$. Unused vehicles are allowed.

`labels = (0, 0, 1, 1)` assigns customers 0 and 1 to vehicle 0,
and customers 2 and 3 to vehicle 1. Labels are **not** visit orders.

With $x_{iv}=1$ when vehicle $v$ serves customer $i$, and pair cost $c_{ij}$:
$$F=\sum_v\sum_{i<j}c_{ij}x_{iv}x_{jv},\qquad
\sum_v x_{iv}=1,\qquad \sum_i d_i x_{iv}\le Q.$$

Minimize the sum of costs within groups, subject to the two rules.
Customer coordinates stay in the input to define $c_{ij}$; we only need
the resulting pair costs for this walkthrough.

### Select inputs

Set `INPUT_FILE` in the parameter cell below to `"seeds_s.json"`, `"seeds_m.json"`, or another compatible JSON file. Use the same setting in all three notebooks. Bare filenames resolve under `track_a/inputs/`; repository-relative and absolute paths also work.
`OUTPUT_DIRECTORY = None` selects the existing `outputs/all_demands/` for the default S file, or `outputs/<input-file-stem>/` for other files. An explicit output path is also supported; mismatched saved inputs are rejected.

In [1]:
INPUT_FILE = "seeds_s.json"
OUTPUT_DIRECTORY = "track_a/outputs/seeds_s_comparison_full"
shots, seed = 256, 0

unbalanced_lambda1 = None  # Saved presets: S 0.2; M 0.75. Set a number to override.
unbalanced_lambda2 = None  # Saved presets: S 0.01; M 0.02. Set a number to override.
unbalanced_assignment_penalty = None

variable_reduction_alpha = None
variable_reduction_beta = None


In [2]:
from pathlib import Path
from collections import Counter
from itertools import combinations, product
from time import perf_counter
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "track_a" / "common" / "instances.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from track_a.common.instances import Instance, feasible, notebook_inputs
from track_a.common.models import Qubo, assignment_view, build_qubo
from track_a.common.classical import (
    ANNEALING_STEPS, INITIAL_TEMPERATURE_FACTOR, FINAL_TEMPERATURE_FACTOR,
    cplex_crosscheck, installed_qubo_crosscheck,
)

input_path, output, instances = notebook_inputs(INPUT_FILE, OUTPUT_DIRECTORY)
print("Input:", input_path, "| output:", output)
print("Tier:", instances[0].tier.upper(), "| N, K, Q:",
      instances[0].n, instances[0].vehicles, instances[0].capacity)
display(pd.DataFrame([
    {"instance": i.id, "demands": i.demands, "certificate": i.certificate, **i.ratios}
    for i in instances
]))

Input: /workspaces/duquantum_hack_2026/track_a/inputs/seeds_s.json | output: /workspaces/duquantum_hack_2026/track_a/outputs/seeds_s_comparison_full
Tier: S | N, K, Q: 4 2 7


,instance,demands,certificate,rho,largest_demand_ratio
0,s-0,"(2, 3, 4, 2)","(0, 0, 1, 0)",0.785714,0.571429
1,s-1,"(3, 4, 2, 2)","(0, 0, 1, 1)",0.785714,0.571429
2,s-2,"(1, 2, 3, 4)","(0, 0, 0, 1)",0.714286,0.571429


## 1. Score a grouping

Add distances only between customers sharing a vehicle.
`canonical` renames vehicles in order of first appearance:
`(1, 1, 0, 0)` becomes `(0, 0, 1, 1)`. This avoids counting the same
grouping twice. The input certificate proves feasibility, not optimality.

In [3]:
def objective(instance: Instance, labels: tuple[int, ...] | list[int]) -> float:
    distances = instance.distances
    return float(sum(distances[i, j] for i in range(instance.n)
                     for j in range(i) if labels[i] == labels[j]))


def canonical(labels: tuple[int, ...] | list[int]) -> tuple[int, ...]:
    mapping: dict[int, int] = {}
    return tuple(mapping.setdefault(k, len(mapping)) for k in labels)

## 2. List the possible groupings

Enumerate labeled assignments and keep one representative per vehicle relabeling.
S has $2^4=16$ labeled assignments; M has $3^{10}=59,049$.
The shared experiment stage generates only distinct partitions. Capacity is
checked independently. No L-tier execution is enabled.

In [4]:
def partitions(n: int, vehicles: int):
    for labels in product(range(vehicles), repeat=n):
        if labels == canonical(labels):
            yield labels

## 3. Find the exact clustering optimum

Score every feasible grouping and keep the minimum. Count tied optima
within $10^{-8}$, without double-counting vehicle relabelings.
Independently solve the target QUBO with CPLEX and check its decoded
grouping has the same clustering cost. There is no routing in this search.

In [5]:
def clustering_reference(instance: Instance) -> dict:
    """Exact clustering only; routing references belong to the results stage."""
    start = perf_counter()
    candidates = [labels for labels in partitions(instance.n, instance.vehicles)
                  if feasible(instance, labels)]
    if not candidates:
        raise ValueError("No feasible grouping")
    costs = [objective(instance, labels) for labels in candidates]
    best = min(costs)
    result = {
        "clustering": {"cost": best, "labels": candidates[costs.index(best)],
                       "distinct_optima": sum(abs(cost - best) <= 1e-8 for cost in costs),
                       "status": "proven_optimum"},
        "status": "proven_optimum", "algorithm": "exhaustive-clustering-partitions",
        "settings": {"maximum_customers": instance.n, "vehicle_labels_quotiented": True,
                     "equality_tolerance": 1e-8},
        "feasible_partitions": len(candidates), "seconds": perf_counter() - start,
    }
    if instance.tier == "s":
        result["cplex_crosscheck"] = {
            "clustering": installed_qubo_crosscheck(instance, build_qubo(instance), best),
        }
    else:
        result["cplex_crosscheck"] = cplex_crosscheck(instance, result)
    return result

## 4. Sample the QUBO with classical annealing

Start each shot from random bits. Try one bit flip per step; accept
improvements, and sometimes uphill moves with probability $e^{-\Delta E/T}$.
Cool geometrically from `10 * penalty` to `0.01 * penalty` over 400 steps.
Chains run together for efficiency. No warm start or repair.
The [quantum notebook](quantum_capacity_constrained_clustering.ipynb) derives the QUBO.

In [6]:
def anneal(qubo: Qubo, shots: int, seed: int, steps: int = ANNEALING_STEPS) -> Counter[str]:
    """Independent Metropolis chains, no repair or certificate warm start."""
    rng = np.random.default_rng(seed)
    size = len(qubo.variable_order)
    z = rng.integers(0, 2, size=(shots, size))
    diagonal = np.diag(qubo.matrix)
    pairs = 2 * (qubo.matrix - np.diag(diagonal))
    fields = z @ pairs + diagonal
    temperatures = np.geomspace(INITIAL_TEMPERATURE_FACTOR * qubo.penalty,
                                FINAL_TEMPERATURE_FACTOR * qubo.penalty, steps)
    rows = np.arange(shots)
    for temperature in temperatures:
        columns = rng.integers(size, size=shots)
        change = 1 - 2 * z[rows, columns]
        delta = change * fields[rows, columns]
        accepted = (delta <= 0) | (np.log(rng.random(shots)) < -delta / temperature)
        selected_rows, selected_cols = rows[accepted], columns[accepted]
        fields[selected_rows] += change[accepted, None] * pairs[selected_cols]
        z[selected_rows, selected_cols] = 1 - z[selected_rows, selected_cols]
    return Counter("".join(map(str, row)) for row in z)

## 5. Run the three classical comparisons

Save exact references, exhaustive QUBO checks, configurations and raw counts.
Compare exact constrained search, QUBO annealing and uniform random bits.
All use 256 shots and seed 0. The exact solver's one answer is repeated
256 times; those are **not independent samples**.

In [7]:
from track_a.common.experiments import classical_runs, prepare_experiment

In [8]:
instances = prepare_experiment(output, input_path)
classical_runs(instances, output, shots, seed)
references = {i.id: json.loads((output / "references" / f"{i.id}.json").read_text())
              for i in instances}
display(pd.DataFrame([
    {"instance": name,
     **assignment_view(next(i for i in instances if i.id == name), ref["clustering"]["labels"]),
     "model_cost": ref["clustering"]["cost"],
     "status": ref["status"],
     "model_optima": ref["clustering"]["distinct_optima"]}
    for name, ref in references.items()
]))

Classical s-0
Classical s-1
Classical s-2


,instance,customer_labels,canonical_assignment_bits,model_cost,status,model_optima
0,s-0,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,1
1,s-1,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,1
2,s-2,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,1


## 6. Worked S example and live hand-check of the selected input

The benchmark above runs all selected vectors. The fixed numbers below describe
`seeds_s.json` only; the following code recalculates for the selected tier.
For the S example `s-0`:
demands **[2, 3, 4, 2]**, two vehicles, capacity **7**.
Customer numbers start at **0**.

1. Use the six pair costs in the table below.
2. Check all eight distinct groupings. Sum each group's demands; reject it
   if either load exceeds 7. Exactly four groupings are feasible here.
3. For each feasible grouping, sum costs of pairs in the **same** group.
   Pick the smallest total.

The winning groups are **{0, 3} and {1, 2}**:
- Loads: $2+2=4$ and $3+4=7$; both fit.
- Pair costs: $c_{03}\approx9.570242$ and $c_{12}\approx71.316611$.
- Clustering cost: $9.570242+71.316611\approx\mathbf{80.886853}$.

This completely checks the clustering answer; no visiting order or depot
tour is needed. The tables recalculate if you edit the input, while the
worked numbers above describe the shipped input.

### Common comparison format

Both notebooks display `customer_labels` (one vehicle number per customer)
and `canonical_assignment_bits` (one bit per customer/vehicle pair):

| `customer_labels` | `canonical_assignment_bits` |
| --- | --- |
| `[0, 1, 1, 0]` | `10010110` |

Grouped by customer: `10010110` = `10 | 01 | 01 | 10`.

These are two encodings of the **same** groups, `{0, 3}` and `{1, 2}`.
`customer_labels` is a label list, not a four-bit QUBO string.
Vehicle names are normalized by first appearance, so swapping vehicle 0 and 1
does not make an equivalent grouping look different. Compare `model_cost`
for solution quality. Raw quantum assignment and slack bits remain separate;
they are never rewritten or repaired for this display.


In [9]:
manual_instance = instances[0]
print("Live hand-check:", manual_instance.id)
manual_reference = references[manual_instance.id]
display(pd.DataFrame({"customer": range(manual_instance.n), "demand": manual_instance.demands}))
display(pd.DataFrame([
    {"pair": (i, j), "cost": manual_instance.distances[i, j]}
    for i, j in combinations(range(manual_instance.n), 2)
]).round(6))

manual_candidates = []
for labels in partitions(manual_instance.n, manual_instance.vehicles):
    groups = [tuple(i for i, v in enumerate(labels) if v == vehicle)
              for vehicle in range(manual_instance.vehicles)]
    manual_candidates.append({
        **assignment_view(manual_instance, labels), "groups": groups,
        "loads": [sum(manual_instance.demands[i] for i in group) for group in groups],
        "feasible": feasible(manual_instance, labels),
        "model_cost": objective(manual_instance, labels),
    })
candidate_table = pd.DataFrame(manual_candidates).round(6)
print("Distinct groupings examined:", len(candidate_table))
display(candidate_table if manual_instance.tier == "s" else candidate_table.head(25))
manual_labels = tuple(manual_reference["clustering"]["labels"])
display(pd.DataFrame([{
    "method": "classical-exact", **assignment_view(manual_instance, manual_labels),
    "model_cost": objective(manual_instance, manual_labels),
}]))

Live hand-check: s-0


,customer,demand
0,0,2
1,1,3
2,2,4
3,3,2


,pair,cost
0,"(0, 1)",46.262020
1,"(0, 2)",46.548088
2,"(0, 3)",9.570242
3,"(1, 2)",71.316611
4,"(1, 3)",51.722215
5,"(2, 3)",37.671375


Distinct groupings examined: 8


,customer_labels,canonical_assignment_bits,groups,loads,feasible,model_cost
0,"[0, 0, 0, 0]",10101010,"[(0, 1, 2, 3), ()]","[11, 0]",False,263.090550
1,"[0, 0, 0, 1]",10101001,"[(0, 1, 2), (3,)]","[9, 2]",False,164.126719
2,"[0, 0, 1, 0]",10100110,"[(0, 1, 3), (2,)]","[7, 4]",True,107.554477
3,"[0, 0, 1, 1]",10100101,"[(0, 1), (2, 3)]","[5, 6]",True,83.933395
4,"[0, 1, 0, 0]",10011010,"[(0, 2, 3), (1,)]","[8, 3]",False,93.789704
5,"[0, 1, 0, 1]",10011001,"[(0, 2), (1, 3)]","[6, 5]",True,98.270303
6,"[0, 1, 1, 0]",10010110,"[(0, 3), (1, 2)]","[4, 7]",True,80.886853
7,"[0, 1, 1, 1]",10010101,"[(0,), (1, 2, 3)]","[2, 9]",False,160.710201


,method,customer_labels,canonical_assignment_bits,model_cost
0,classical-exact,"[0, 1, 1, 0]",10010110,80.886853


## Additional solver: slack-free unbalanced penalization

This is **added alongside** the binary-slack QUBO, not a replacement.
Following the photo's "Optimize H", define each vehicle's capacity headroom as

$$h_k(x)=Q-\sum_{j=0}^{N-1}d_jx_{jk},\qquad h_k(x)\geq0.$$

The assignment variable $x_{jk}\in\{0,1\}$ is 1 when customer $j$ uses vehicle
$k$; $d_j$ is that customer's demand, $Q$ the capacity, $N$ the customer count,
and $K$ the vehicle count. Capacity equality is allowed. The sums in $h_k$
include $x_{jk}$: demand only counts when assigned to this vehicle.

$$H_u(x)=\frac{F(x)}{s}
+\beta\sum_j\left(\sum_kx_{jk}-1\right)^2
+\sum_k\left[-\lambda_1h_k(x)+\lambda_2h_k(x)^2\right].$$

Here $F$ is the same within-vehicle pair-distance cost as before
($\frac12x^\mathsf{T}Dx$ when $D$ repeats the distance matrix by vehicle),
$s=\max(1,\max_{ij}c_{ij})$ is the existing distance normalization, $\beta$ is
the assignment penalty, and $\lambda_1,\lambda_2>0$ are tunable capacity
weights. The matrix convention remains $H_u=x^\mathsf{T}Q_ux+c_u$.
With $L_k=\sum_jd_jx_{jk}$, the capacity term expands to

$$-\lambda_1Q+\lambda_2Q^2
+(\lambda_1-2\lambda_2Q)L_k+\lambda_2L_k^2.$$

Only $NK$ assignment bits are used: **S 8, M 30**, versus **S 14, M 45**
with binary slack. There are no slack or auxiliary bits.

The parameter cell exposes `unbalanced_lambda1`, `unbalanced_lambda2`, and
`unbalanced_assignment_penalty`. `None` for either lambda selects tier-aware
saved presets in `inputs/unbalanced_penalties.json`: **S 0.2 and 0.01**;
**M 0.75 and 0.02**. Numeric overrides remain available.
M was selected from an 80-pair SA search and fresh-seed checks on the
current three M inputs, prioritizing best feasible clustering gap rather
than raw feasibility. It is best-tested, not globally optimal; see the
saved evidence in `outputs/penalty_tuning/m_unbalanced_search.json`.
The S weights preserve the exact constrained optimum in all 256-bitstring
checks per instance and improve measured SA optimal-hit rates. This is
benchmark tuning, not held-out instance validation or a general guarantee.
`None` for the assignment penalty uses the same
certificate-based numeric value as the original solver, but **not its proof**.
The certificate is never a warm start. Use identical weights in both notebooks.

Unlike the original slack penalty, this term is not zero on every feasible
assignment: its minimum occurs at $h=\lambda_1/(2\lambda_2)$. It can change the
preferred feasible grouping, and its ground state can violate constraints.
We therefore check original one-hot and capacity constraints for every raw
sample; no repair is applied. Compare **`model_cost`**, feasibility rates and
gaps to the constrained reference, not $H_u$ against the slack-QUBO energy.


In [10]:
from track_a.common.models import build_unbalanced_qubo, unbalanced_penalty_weights
from track_a.common.unbalanced import classical_unbalanced_runs, comparison_rows as solver_comparison_rows

effective_lambda1, effective_lambda2 = unbalanced_penalty_weights(
    instances[0], unbalanced_lambda1, unbalanced_lambda2)
print("Effective unbalanced weights:", effective_lambda1, effective_lambda2)

# Check the matrix expansion on a feasible assignment, without using it to initialize SA.
unbalanced_checks = []
for instance in instances:
    target = build_unbalanced_qubo(instance, unbalanced_lambda1, unbalanced_lambda2,
                                   unbalanced_assignment_penalty)
    labels = instance.certificate
    x = np.array([[int(labels[j] == k) for k in range(instance.vehicles)]
                  for j in range(instance.n)])
    h = instance.capacity - np.array(instance.demands) @ x
    direct_energy = (objective(instance, labels) / target.normalization
                     + target.penalty * np.sum((x.sum(axis=1) - 1)**2)
                     + np.sum(-effective_lambda1 * h + effective_lambda2 * h**2))
    matrix_energy = target.energy(x.ravel())
    assert np.isclose(direct_energy, matrix_energy)
    unbalanced_checks.append({"instance": instance.id, "bits": len(target.variable_order),
                              "slack_bits": target.slack_bits, "headroom": h.tolist(),
                              "formula_energy": direct_energy, "matrix_energy": matrix_energy})
display(pd.DataFrame(unbalanced_checks))

classical_unbalanced_runs(
    instances, output, shots, seed,
    lambda1=unbalanced_lambda1, lambda2=unbalanced_lambda2,
    assignment_penalty=unbalanced_assignment_penalty,
)
display(pd.DataFrame(solver_comparison_rows(instances, output, seed)))
print("Unbalanced raw samples saved in:", output / "samples")


Effective unbalanced weights: 0.2 0.01


,instance,bits,slack_bits,headroom,formula_energy,matrix_energy
0,s-0,8,0,"[0, 3]",0.998127,0.998127
1,s-1,8,0,"[0, 3]",0.666912,0.666912
2,s-2,8,0,"[1, 3]",1.601381,1.601381


Classical unbalanced s-0: 8 bits
Classical unbalanced s-1: 8 bits
Classical unbalanced s-2: 8 bits


,instance,method,encoding,customer_labels,canonical_assignment_bits,model_cost,status,outcome,bits,actual_shots,feasible_shots,feasibility_rate,best_model_gap,error,lambda1,lambda2,assignment_penalty
0,s-0,classical-exact,constrained,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,Exact constrained optimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s-0,classical-qubo-sa,binary-slack-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,14.0,256.0,55.0,0.214844,0.000000,NaN,NaN,NaN,2.508127
2,s-0,classical-unbalanced-sa,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,219.0,0.855469,0.000000,NaN,0.2,0.01,2.508127
3,s-0,quantum-global,binary-slack-v1,"[0, 0, 1, 1]",10100101,83.933395,completed,Feasible sample found,14.0,256.0,7.0,0.027344,3.766423,NaN,NaN,NaN,2.508127
4,s-0,quantum-unbalanced,unbalanced-v1,"[0, 0, 1, 1]",10100101,83.933395,completed,Feasible sample found,8.0,256.0,5.0,0.019531,3.766423,NaN,1.0,0.10,2.508127
5,s-1,classical-exact,constrained,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,Exact constrained optimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,s-1,classical-qubo-sa,binary-slack-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,14.0,256.0,51.0,0.199219,0.000000,NaN,NaN,NaN,2.176912
7,s-1,classical-unbalanced-sa,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,256.0,1.000000,0.000000,NaN,0.2,0.01,2.176912
8,s-1,quantum-global,binary-slack-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,14.0,256.0,5.0,0.019531,0.000000,NaN,NaN,NaN,2.176912
9,s-1,quantum-unbalanced,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,4.0,0.015625,0.000000,NaN,1.0,0.10,2.176912


Unbalanced raw samples saved in: /workspaces/duquantum_hack_2026/track_a/outputs/seeds_s_comparison/samples


## Additional QUBO: variable_reduction (photographed formula)

As requested, preserve the photo's **minus binary-slack sign**:

$$H_v=F(x)/s+\alpha\sum_k\left(\frac{L_k(x)}{Q}-1-b_k\right)^2
+\beta\sum_j\left(\sum_kx_{jk}-1\right)^2,\quad
L_k(x)=\sum_jd_jx_{jk},\quad b_k\in\{0,1\}.$$

$x_{jk}$ assigns customer $j$ to vehicle $k$, $d_j$ is demand, $Q$ is vehicle
capacity, and $F/s$ is the same normalized compactness objective as before.
$\alpha$ weights normalized capacity residuals and $\beta$ assignment residuals.
The demand sum explicitly includes $x_{jk}$, which is implicit in the photo.

**This retains one binary slack bit per vehicle**, rather than eliminating all
slack: **S 10 bits, M 33 bits**, compared with slack 14/45 and unbalanced 8/30.
It is not an exact encoding of $L_k\le Q$: zero capacity penalty occurs at
$L_k=Q$ or $2Q$, so overcapacity can be favored. Original constraints are checked
independently, without repairing samples. We do not change the photographed
minus sign into the usual positive-slack equality.

Direct expansion gives negative assignment/slack couplings, unsupported by the
repulsive quantum embedding. Use the exact reversible substitution
$b_k=1-z_k$, $z_k\in\{0,1\}$:

$$\left(L_k/Q-1-b_k\right)^2=\left(L_k/Q+z_k-2\right)^2.$$

Both classical and quantum run this same bijectively transformed QUBO.
The physical bit order is customer-major assignment bits followed by
`z_0, ..., z_(K-1)`. Evidence retains the raw physical bits and the logical
`b_k = 1-z_k` values. Complementing auxiliaries is not sample repair.

`variable_reduction_alpha` and `variable_reduction_beta` are independently
configurable. `None` sets each to `1 + certificate_cost / s`, an **untuned
starting value, not a feasibility proof**. Energies of different formulations
are not directly comparable; use original feasible costs, gaps and raw-shot
success rates. The previous solvers and their saved results remain intact.


In [3]:
from track_a.common.models import build_variable_reduction_qubo, variable_reduction_penalties, objective as photo_objective
from track_a.common.variable_reduction import classical_variable_reduction_runs
from track_a.common.unbalanced import comparison_rows as solver_comparison_rows

photo_checks = []
for instance in instances:
    alpha, beta = variable_reduction_penalties(
        instance, variable_reduction_alpha, variable_reduction_beta)
    target = build_variable_reduction_qubo(instance, alpha, beta)
    x = np.array([[int(instance.certificate[j] == k) for k in range(instance.vehicles)]
                  for j in range(instance.n)])
    logical_b = np.zeros(instance.vehicles, dtype=int)
    physical_z = 1 - logical_b
    loads = np.array(instance.demands) @ x
    direct_energy = (photo_objective(instance, instance.certificate) / target.normalization
                     + alpha * np.sum((loads / instance.capacity - 1 - logical_b)**2)
                     + beta * np.sum((x.sum(axis=1) - 1)**2))
    matrix_energy = target.energy(np.concatenate([x.ravel(), physical_z]))
    assert np.isclose(direct_energy, matrix_energy)
    photo_checks.append({"instance": instance.id, "bits": len(target.variable_order),
                         "alpha": alpha, "beta": beta, "photo_energy": direct_energy,
                         "matrix_energy": matrix_energy})
display(pd.DataFrame(photo_checks))
classical_variable_reduction_runs(
    instances, output, shots, seed, alpha=variable_reduction_alpha, beta=variable_reduction_beta)
display(pd.DataFrame(solver_comparison_rows(instances, output, seed)))
print("Variable-reduction samples:", output / "samples")


,instance,bits,alpha,beta,photo_energy,matrix_energy
0,s-0,10,2.508127,2.508127,1.968803,1.968803
1,s-1,10,2.176912,2.176912,1.576753,1.576753
2,s-2,10,3.301381,3.301381,2.975133,2.975133


classical-variable_reduction-sa s-0: 10 bits


classical-variable_reduction-sa s-1: 10 bits


classical-variable_reduction-sa s-2: 10 bits


,instance,method,encoding,customer_labels,canonical_assignment_bits,model_cost,status,outcome,bits,actual_shots,feasible_shots,feasibility_rate,best_model_gap,error,lambda1,lambda2,alpha,beta,assignment_penalty
0,s-0,classical-exact,constrained,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,Exact constrained optimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s-0,classical-qubo-sa,binary-slack-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,14.0,256.0,55.0,0.214844,0.000000,NaN,NaN,NaN,NaN,NaN,2.508127
2,s-0,classical-unbalanced-sa,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,219.0,0.855469,0.000000,NaN,0.2,0.01,NaN,NaN,2.508127
3,s-0,classical-variable_reduction-sa,variable-reduction-complemented-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,10.0,256.0,220.0,0.859375,0.000000,NaN,NaN,NaN,2.508127,2.508127,2.508127
4,s-0,quantum-global,binary-slack-v1,"[0, 0, 1, 1]",10100101,83.933395,completed,Feasible sample found,14.0,256.0,7.0,0.027344,3.766423,NaN,NaN,NaN,NaN,NaN,2.508127
5,s-0,quantum-unbalanced,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,4.0,0.015625,0.000000,NaN,0.2,0.01,NaN,NaN,2.508127
6,s-1,classical-exact,constrained,"[0, 1, 1, 0]",10010110,80.886853,proven_optimum,Exact constrained optimum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,s-1,classical-qubo-sa,binary-slack-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,14.0,256.0,51.0,0.199219,0.000000,NaN,NaN,NaN,NaN,NaN,2.176912
8,s-1,classical-unbalanced-sa,unbalanced-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,8.0,256.0,256.0,1.000000,0.000000,NaN,0.2,0.01,NaN,NaN,2.176912
9,s-1,classical-variable_reduction-sa,variable-reduction-complemented-v1,"[0, 1, 1, 0]",10010110,80.886853,completed,Feasible sample found,10.0,256.0,232.0,0.906250,0.000000,NaN,NaN,NaN,2.176912,2.176912,2.176912


Variable-reduction samples: /workspaces/duquantum_hack_2026/track_a/outputs/seeds_s_comparison_full/samples


Next: [quantum clustering](quantum_capacity_constrained_clustering.ipynb).
The required CVRP reference and fixed-group route evaluation appear only in
[results](results.ipynb), not in the clustering optimization.

All supplied demand vectors are run. Inputs include validated feasibility
witnesses, but are explicit rows rather than Track A's seed-based generator (A2).